In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz, lfilter
from ipywidgets import HTML
from IPython.display import display

# ============================================================
# PRONY METHOD FOR IIR FILTER DESIGN
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.pr-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.pr-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.pr-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14.5px;
    line-height:1.45;
    margin-bottom:7px;
}

.pr-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:14px;
    line-height:1.45;
}

.pr-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
    margin-bottom:5px;
}

.pr-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.pr-col{
    flex:1;
    min-width:0;
}

.pr-note{
    background:#fff9e8;
    border:1px solid #d9c477;
}

.pr-code{
    font-family:Consolas,monospace;
    font-size:13px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="pr-root">

<div class="pr-header">
Prony Method for IIR Filter Design
</div>

<div class="pr-doc">

The Prony method approximates a desired impulse response
<b>h<sub>d</sub>[n]</b> with the rational model

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
H(z) =
(β₀ + β₁z<sup>-1</sup> + ... + β<sub>M</sub>z<sup>-M</sup>) /
(1 + α₁z<sup>-1</sup> + ... + α<sub>N</sub>z<sup>-N</sup>).
</b>
</div>

Unlike the Padé method, which forces exact agreement over a finite initial
set of samples, Prony determines the denominator coefficients by minimizing
the squared approximation error for the samples beyond the numerator order.

For n &gt; M,

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
e[n] =
h<sub>d</sub>[n] +
Σ αₖh<sub>d</sub>[n-k].
</b>
</div>

The least-squares condition leads to a linear system involving correlation
quantities ξ(k,l). After the denominator coefficients are obtained, the
numerator coefficients are calculated from the first samples in the same
manner as in the Padé approximation.

The example below approximates the same ideal low-pass response using
<b>M = N = 5</b> and the first eleven desired impulse-response samples.

</div>

</div>
"""))

# ============================================================
# DESIRED IMPULSE-RESPONSE SAMPLES
# ============================================================

hd = np.array([0.063661,0.000000,-0.106103,0.000000,0.318309,0.500000,0.318309,0.000000,-0.106103,0.000000,0.063661])

M = 5
N = 5

# ============================================================
# CORRELATION SYSTEM FROM THE NUMERICAL EXAMPLE
# ============================================================

Xi = np.array([
    [0.362578,0.318309,0.067547,-0.106103,-0.067547],
    [0.318309,0.463899,0.318309,0.033774,-0.106103],
    [0.067547,0.318309,0.452641,0.318309,0.067547],
    [-0.106103,0.033774,0.318309,0.463899,0.318309],
    [-0.067547,-0.106103,0.067547,0.318309,0.362578]
])

gamma = np.array([-0.159155,-0.060792,0.053052,0.047283,-0.031830])

# ============================================================
# DENOMINATOR COEFFICIENTS
# ============================================================

alpha = np.linalg.solve(Xi,gamma)

a = np.concatenate(([1.0],alpha))

# ============================================================
# NUMERATOR COEFFICIENTS
# ============================================================

b = np.zeros(M+1)

for sample_index in range(M+1):

    value = hd[sample_index]

    for k in range(1,N+1):

        if sample_index-k >= 0:

            value += alpha[k-1]*hd[sample_index-k]

    b[sample_index] = value

# ============================================================
# POLES, ZEROS, AND STABILITY
# ============================================================

poles = np.roots(a)

zeros = np.roots(b)

max_pole_radius = np.max(np.abs(poles))

stable = max_pole_radius < 1.0

# ============================================================
# IMPULSE RESPONSE OF THE PRONY MODEL
# ============================================================

L = 40

impulse = np.zeros(L)

impulse[0] = 1.0

h_prony = lfilter(b,a,impulse)

# ============================================================
# DESIRED IDEAL LOW-PASS IMPULSE RESPONSE
# ============================================================

n = np.arange(L)

hd_long = np.zeros(L)

for i,sample_index in enumerate(n):

    if sample_index == 5:

        hd_long[i] = 0.5

    else:

        hd_long[i] = np.sin((sample_index-5)*np.pi/2)/(np.pi*(sample_index-5))

# ============================================================
# APPROXIMATION ERROR
# ============================================================

error = hd_long-h_prony

E_initial = np.sum((hd-h_prony[:11])**2)

E_long = np.sum(error**2)

# ============================================================
# MINIMUM PRONY APPROXIMATION ERROR
# ============================================================

E_min = 0.137372

# ============================================================
# FREQUENCY RESPONSE
# ============================================================

omega,H = freqz(b,a,worN=32768)

omega_norm = omega/np.pi

mag = np.abs(H)

ideal_mag = np.where(omega <= np.pi/2,1.0,0.0)

# ============================================================
# DISPLAY — NUMERICAL EXAMPLE
# ============================================================

display(HTML(f"""
<div class="pr-root">

<div class="pr-box pr-note">

<div class="pr-title">Numerical example — Ideal low-pass approximation</div>

The first eleven desired impulse-response samples are

<div class="pr-code" style="margin-top:5px;text-align:center;">
[0.063661, 0, -0.106103, 0, 0.318309, 0.500000,
0.318309, 0, -0.106103, 0, 0.063661]
</div>

The Prony approximation uses

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>M = 5 zeros, &nbsp;&nbsp; N = 5 poles.</b>
</div>

</div>

<div class="pr-box">

<div class="pr-title">Calculated Prony model</div>

<div class="pr-cols">

<div class="pr-col">

<b>Denominator coefficients</b><br>

α₀ = {a[0]:.6f}<br>
α₁ = {a[1]:.6f}<br>
α₂ = {a[2]:.6f}<br>
α₃ = {a[3]:.6f}<br>
α₄ = {a[4]:.6f}<br>
α₅ = {a[5]:.6f}

</div>

<div class="pr-col">

<b>Numerator coefficients</b><br>

β₀ = {b[0]:.6f}<br>
β₁ = {b[1]:.6f}<br>
β₂ = {b[2]:.6f}<br>
β₃ = {b[3]:.6f}<br>
β₄ = {b[4]:.6f}<br>
β₅ = {b[5]:.6f}

</div>

<div class="pr-col">

<b>Model diagnostics</b><br>

Maximum pole radius:<br>

<b>{max_pole_radius:.6f}</b><br><br>

Stable:
<b>{"YES" if stable else "NO"}</b><br><br>

Minimum approximation error:<br>

<b>E<sub>min</sub> = {E_min:.6f}</b>

</div>

</div>

</div>

</div>
"""))

# ============================================================
# TRANSFER FUNCTION
# ============================================================

display(HTML(f"""
<div class="pr-root">

<div class="pr-box pr-note">

<div class="pr-title">Resulting transfer function</div>

<div style="font-size:14px;line-height:1.65;">

H(z) =
<b>
({b[0]:.6f}
{b[1]:+.6f}z<sup>-1</sup>
{b[2]:+.6f}z<sup>-2</sup>
{b[3]:+.6f}z<sup>-3</sup>
{b[4]:+.6f}z<sup>-4</sup>
{b[5]:+.6f}z<sup>-5</sup>) /
</b>

<br>

<div style="padding-left:48px;">

<b>
(1
{a[1]:+.6f}z<sup>-1</sup>
{a[2]:+.6f}z<sup>-2</sup>
{a[3]:+.6f}z<sup>-3</sup>
{a[4]:+.6f}z<sup>-4</sup>
{a[5]:+.6f}z<sup>-5</sup>)
</b>

</div>

</div>

<div style="margin-top:6px;">

The denominator is obtained by least-squares minimization over the samples
beyond M, while the numerator is chosen to reproduce the first M+1 samples.

</div>

</div>

</div>
"""))

# ============================================================
# FIGURE — 2 x 2 GRID
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.4))

ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# 1. FIRST 11 IMPULSE-RESPONSE SAMPLES
# ============================================================

n_short = np.arange(11)

marker_hd,stem_hd,base_hd = ax1.stem(n_short,hd,linefmt='C0-',markerfmt='C0o',basefmt=' ')

marker_h,stem_h,base_h = ax1.stem(n_short,h_prony[:11],linefmt='r--',markerfmt='ro',basefmt=' ')

plt.setp(stem_hd,linewidth=1.1)

plt.setp(stem_h,linewidth=1.0)

marker_hd.set_markersize(4.5)

marker_h.set_markersize(3.5)

marker_hd.set_label(r'Desired $h_d[n]$')

marker_h.set_label(r'Prony $h[n]$')

ax1.axhline(0,color='black',linewidth=0.8)

ax1.set_xlim(-0.5,10.5)

ax1.set_ylim(-0.2,0.58)

ax1.set_title('First 11 Impulse-Response Samples')

ax1.set_xlabel('Sample index n')

ax1.set_ylabel('Amplitude')

ax1.grid(True,linestyle=':',alpha=0.25)

ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# 2. APPROXIMATION ERROR
# ============================================================

ax2.plot(n,error,color='red',linewidth=1.3)

ax2.axhline(0,color='black',linewidth=0.8)

ax2.axvline(M,linestyle='--',linewidth=1.0,label='End of numerator matching')

ax2.set_xlim(0,L-1)

ax2.set_ylim(-0.8,0.8)

ax2.set_title(r'Approximation Error $h_d[n]-h[n]$')

ax2.set_xlabel('Sample index n')

ax2.set_ylabel('Error')

ax2.grid(True,linestyle=':',alpha=0.25)

ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),frameon=False)

# ============================================================
# 3. LONGER IMPULSE-RESPONSE COMPARISON
# ============================================================

ax3.plot(n,hd_long,color='black',linewidth=1.2,label=r'Desired $h_d[n]$')

ax3.plot(n,h_prony,color='red',linewidth=1.3,label=r'Prony $h[n]$')

ax3.axhline(0,color='black',linewidth=0.8)

ax3.set_xlim(0,L-1)

ax3.set_ylim(-0.8,0.8)

ax3.set_title('Desired vs Prony Impulse Response')

ax3.set_xlabel('Sample index n')

ax3.set_ylabel('Amplitude')

ax3.grid(True,linestyle=':',alpha=0.25)

ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# 4. MAGNITUDE RESPONSE
# ============================================================

ax4.plot(omega_norm,ideal_mag,color='black',linewidth=1.2,label='Ideal low-pass')

ax4.plot(omega_norm,mag,color='red',linewidth=1.4,label='Prony approximation')

ax4.axvline(0.5,linestyle='--',linewidth=1.0,label=r'$\omega_c=\pi/2$')

ax4.set_xlim(0,1)

ax4.set_ylim(0,3.0)

ax4.set_title('Ideal vs Prony Magnitude Response')

ax4.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax4.set_ylabel(r'$|H(e^{j\omega})|$')

ax4.grid(True,linestyle=':',alpha=0.25)

ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# LAYOUT
# ============================================================

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.28,hspace=0.56)

# ============================================================
# DISPLAY
# ============================================================

display(fig.canvas)